# Copia pública del 22 de octubre de 2026

Código de investigación; revisar entradas en `../README.md`. Las salidas guardadas se retiraron. Las etapas 01–06 no se reejecutaron para esta publicación. Para la síntesis numérica utilizada en la memoria, ejecutar `07_Sintesis_resultados_20261022.ipynb`. Las utilidades interactivas son opcionales de revisión y no una dependencia de la síntesis científica.

In [ ]:
from pathlib import Path
import sys
BASE = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'Configuracion.json').is_file()), None)
if BASE is None:
    raise FileNotFoundError('Iniciar Jupyter en notebooks/actual_20261022 o su subcarpeta de notebooks.')
sys.path.insert(0, str(BASE))
import publicacion_config


# 02 · Perfiles y figuras de resultados
Perfiles longitudinales y χ, pruebas de concavidad, sensibilidad de ksn, terrazas y sus cruces, KP manuales, litología y mapas en planta. Los **18 ríos** se evalúan con las mismas entradas; la selección limita únicamente las figuras.

**Preparado sin ejecutar.** Correr 01 antes de este notebook. Las figuras se exportan en PNG de 300 dpi y PDF: `05_Chi_v1`, `09_Longitudinal_v1`, etc., en `Productos/02_Perfiles`. Volver a dibujar crea otra versión de figura. Cambiar parámetros científicos requiere otra corrida compartida en `Configuracion.json`.

La terraza proyectada por XY y el cruce del río con su cota son observaciones distintas. Un cambio de carta no es un contacto geológico demostrado. Las propuestas aluviales siguen pendientes de revisión.

**Ordenación 19/10/2026:** este notebook describe los 18 ríos; no calcula K ni τ. Sus terrazas publicadas son contexto descriptivo y no reemplazan la asociación por B/MIS del nuevo 03. Se conserva la asignación litológica existente. Se eliminó la validación repetida de toda la cadena de archivos dentro del bucle de sensibilidades, manteniendo la verificación de cada archivo leído.

In [ ]:
from pathlib import Path
import sys
BASE = publicacion_config.base()
if not (BASE/'Configuracion.json').is_file():
    BASE = Path(str(BASE / 'inputs_external' / 'Jupyter_Memoria'))
if not (BASE/'Configuracion.json').is_file():
    raise FileNotFoundError('Abrir Jupyter desde la carpeta Jupyter_Memoria.')
CARPETA_NOTEBOOKS = BASE
CARPETA_CODIGO = BASE/'Archivos_Memoria/Codigo'
sys.path.insert(0,str(CARPETA_CODIGO))
import importlib
import memoria_comun as pa
pa=importlib.reload(pa)
CFG,RUN_ID=pa.CFG,pa.RUN_ID
sys.path.insert(0,str(Path(CFG['topoanalysis_codigo']).parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'pdf.fonttype':42,'font.family':'DejaVu Sans','font.size':10})
print('Corrida compartida:',RUN_ID,'| θ:',CFG['theta'],'| ventana:',CFG['ventana_m'],'m')

pa.NOTEBOOK_PATH=BASE/'Notebooks_principales_20261019/02_Perfiles.ipynb'
from scipy.integrate import cumulative_trapezoid
from scipy.stats import linregress
perfiles,NOMBRES,MANIFEST_PERFILES=pa.load_profiles()
kp,kp_rechazados=pa.load_manuals(perfiles)
kp=kp.copy();kp['tipo']='manual_reubicado'
for rid,p in perfiles.items():
    p['chi_m']=pa.chi_1d(p)
kp['chi_m']=[np.interp(r.distancia_m,perfiles[r.ID].distancia_m,perfiles[r.ID].chi_m) for _,r in kp.iterrows()]
THETA,A0=CFG['theta'],CFG['A0_m2']
RIOS_VER=list(perfiles)  # selección exclusivamente gráfica
GUARDAR_FIGURAS=True
FUENTES={}
def registrar_archivo(path):
    FUENTES[str(path)]=pa.file_hash(path)
    return FUENTES[str(path)]

## Terrazas: ubicación, proyección, cotas y selección
Se conservan todas las asociaciones de la planilla, las apartadas y todos los cruces con H y H±error publicado. Las coordenadas requieren EPSG:32718. El KP del Excel sirve para contrastar con el CSV manual; nunca lo sustituye.

In [ ]:
nombres,manifest=NOMBRES,MANIFEST_PERFILES
manuales,manuales_rechazados=kp,kp_rechazados
deps = pa.dependency('perfiles')
for key in ['kp_manual_csv', 'terrazas_excel', 'revision_muestras_excel', 'seleccion_excel']:
    deps[CFG[key]] = pa.file_hash(CFG[key])
pa.begin('terrazas')
terrazas_originales = pd.read_excel(CFG['terrazas_excel'], sheet_name='Hoja1')
revision_muestras = pd.read_excel(CFG['revision_muestras_excel'], sheet_name='Resumen')
seleccion_excel = pd.read_excel(CFG['seleccion_excel'], sheet_name='Hoja1').dropna(how='all')
chi_por_rio = {rid: pa.chi_1d(p) for rid,p in perfiles.items()}

# Catálogo descriptivo de cotas publicadas. En 03 H para PT se muestrea en el DEM; no usar estas asociaciones como selección automática de B.
fuente_JM=pd.read_csv(CFG['fuente_JM_csv']).set_index('terrace_id')
terrazas_originales['Elevación Excel original (m)']=terrazas_originales['Elevación (m)']
terrazas_originales['Error Excel original (m)']=terrazas_originales['Error elevación (m)']
terrazas_originales['Fuente cota adoptada']='Excel revisado'
for j,fila in terrazas_originales.iterrows():
    tid=pa.text(fila.Terraza).replace('-','_')
    if tid in fuente_JM.index:
        terrazas_originales.loc[j,'Elevación (m)']=fuente_JM.loc[tid,'elevacion_m']
        terrazas_originales.loc[j,'Error elevación (m)']=fuente_JM.loc[tid,'error_elevacion_m']
        terrazas_originales.loc[j,'Fuente cota adoptada']='Suplemento Jara-Muñoz, ID exacto'
deps[CFG['fuente_JM_csv']]=pa.file_hash(CFG['fuente_JM_csv'])

## Normalizar asociaciones y preparar proyecciones
Una proyección espacial no es un cruce por cota. Se conservan las filas sin asociación, sin coordenadas o con CRS incompatible.

In [ ]:
def proyectar_terraza(p, este, norte):
    xy = p[["x_utm", "y_utm"]].to_numpy(float)
    inicio = xy[:-1]
    vector = xy[1:] - inicio
    longitud2 = np.sum(vector**2, axis=1)

    if np.any(longitud2 <= 0):
        raise ValueError("El cauce contiene segmentos espaciales de longitud cero.")

    punto = np.array([este, norte], dtype=float)
    fraccion = np.clip(
        np.sum((punto - inicio) * vector, axis=1) / longitud2,
        0, 1,
    )

    proyecciones = inicio + fraccion[:, None] * vector
    distancias = np.linalg.norm(proyecciones - punto, axis=1)
    j = int(np.argmin(distancias))
    f = fraccion[j]

    distancia = p["distancia_m"].to_numpy(float)
    elevacion = p["elevacion_m"].to_numpy(float)

    return {
        "distancia_proyectada_m": distancia[j] + f * (
            distancia[j + 1] - distancia[j]
        ),
        "distancia_lateral_m": distancias[j],
        "x_proyectado": proyecciones[j, 0],
        "y_proyectado": proyecciones[j, 1],
        "z_cauce_proyectado_m": elevacion[j] + f * (
            elevacion[j + 1] - elevacion[j]
        ),
    }

In [ ]:
requeridas = {'Cauce', 'Cauce asociado', 'Terraza', 'Elevación (m)', 'Error elevación (m)',
    'Este UTM18S (m)', 'Norte UTM18S (m)', 'CRS coordenadas UTM'}
if requeridas - set(terrazas_originales): raise ValueError(f'Faltan columnas: {requeridas-set(terrazas_originales)}')
nombre_a_id = {pa.normalizar_nombre(v): k for k,v in nombres.items()}
if len(nombre_a_id) != len(nombres): raise ValueError('Nombres normalizados ambiguos.')
t = terrazas_originales.copy()
t['fila_excel'] = np.arange(len(t)) + 2
t['Cauce'] = t['Cauce'].ffill()
t['cauce_excel'] = t['Cauce asociado'].fillna(t['Cauce'])
t['ID'] = t.cauce_excel.map(lambda x: nombre_a_id.get(pa.normalizar_nombre(x)))
t['terraza_id'] = t.Terraza.astype('string').str.strip().str.replace('-', '_', regex=False)
for col in ['Elevación (m)', 'Error elevación (m)', 'Este UTM18S (m)', 'Norte UTM18S (m)']:
    t[col] = pd.to_numeric(t[col], errors='coerce')
filas, apartadas = [], []
for row in t.to_dict('records'):
    try:
        rid = row['ID']
        if rid not in perfiles or pd.isna(row['terraza_id']): raise ValueError('Sin asociación o identificador')
        if '32718' not in str(row['CRS coordenadas UTM']): raise ValueError('CRS de coordenadas no confirmado')
        if not np.isfinite([row[k] for k in ['Elevación (m)', 'Este UTM18S (m)', 'Norte UTM18S (m)']]).all():
            raise ValueError('Cota o coordenadas ausentes')
        row['asociacion_id'] = f"R{int(rid):02d}_{row['terraza_id']}"
        row.update(proyectar_terraza(perfiles[rid], row['Este UTM18S (m)'], row['Norte UTM18S (m)']))
        row['chi_proyectado_m'] = float(np.interp(row['distancia_proyectada_m'], perfiles[rid].distancia_m, chi_por_rio[rid]))
        row['proyeccion_en_extremo'] = bool(np.isclose(row['distancia_proyectada_m'], 0) or np.isclose(row['distancia_proyectada_m'], perfiles[rid].distancia_m.iloc[-1]))
        row['estado'] = 'ok'; filas.append(row)
    except (ValueError, KeyError, TypeError) as exc:
        row.update(estado='pendiente', detalle=str(exc)); apartadas.append(row)
terrazas = pd.DataFrame(filas, columns=[*t.columns, 'asociacion_id', 'distancia_proyectada_m',
    'distancia_lateral_m', 'x_proyectado', 'y_proyectado', 'z_cauce_proyectado_m',
    'chi_proyectado_m', 'proyeccion_en_extremo', 'estado'])
if terrazas.asociacion_id.duplicated().any(): raise ValueError('Asociación río-terraza duplicada; revisar la planilla.')
terrazas_no_asociadas = pd.DataFrame(apartadas, columns=[*t.columns, 'estado', 'detalle'])

## Todos los cruces y sus incertidumbres verticales
Los segmentos planos coincidentes se exportan como intervalos ambiguos. No se elige automáticamente un cruce, ni se interpola a través de huecos de elevación.

In [ ]:
filas_cruces, filas_asociaciones = [], []
for row in terrazas.to_dict('records'):
    rid, p = row['ID'], perfiles[row['ID']]
    cota, error = float(row['Elevación (m)']), row['Error elevación (m)']
    niveles = [('central', cota)]
    if pd.notna(error) and np.isfinite(error) and error >= 0:
        niveles += [('inferior', cota-error), ('superior', cota+error)]
    n_central = n_planos = 0
    for nivel, altura in niveles:
        posiciones, planos = pa.crossings(p, altura)
        for j, s in enumerate(posiciones, 1):
            filas_cruces.append({'ID': rid, 'terraza_id': row['terraza_id'], 'asociacion_id': row['asociacion_id'],
                'nivel': nivel, 'cruce_id': j, 'tipo': 'puntual', 'desde_m': s, 'hasta_m': s,
                'cota_m': altura, 'chi_m': float(np.interp(s, p.distancia_m, chi_por_rio[rid]))})
        for j, (a,b) in enumerate(planos, 1):
            filas_cruces.append({'ID': rid, 'terraza_id': row['terraza_id'], 'asociacion_id': row['asociacion_id'],
                'nivel': nivel, 'cruce_id': j, 'tipo': 'plano_ambiguo', 'desde_m': a, 'hasta_m': b,
                'cota_m': altura, 'chi_m': np.nan})
        if nivel == 'central': n_central, n_planos = len(posiciones), len(planos)
    row.update(n_cruces=n_central, n_planos=n_planos,
        estado_cruces='ambiguo' if n_planos or n_central>1 else 'unico' if n_central==1 else 'sin_cruce',
        perfil_hash=pa.profile_hash(p))
    filas_asociaciones.append(row)
asociaciones = pd.DataFrame(filas_asociaciones, columns=[*terrazas.columns, 'n_cruces', 'n_planos', 'estado_cruces', 'perfil_hash'])
cruces = pd.DataFrame(filas_cruces, columns=['ID','terraza_id','asociacion_id','nivel','cruce_id','tipo','desde_m','hasta_m','cota_m','chi_m'])

## Selección vigente y discrepancias Excel–CSV
Se parte de los 18 ríos. El ID de knickpoint del Excel solo se conserva como comparación; nunca reemplaza al CSV manual.

In [ ]:
if {'id','Cauce','Terraza','ID knickpoint'} - set(seleccion_excel):
    raise ValueError('Faltan columnas de selección en el Excel.')
seleccion_excel['ID'] = seleccion_excel['id'].map(pa.normalize_id)
if seleccion_excel.ID.duplicated().any(): raise ValueError('Más de una selección por río en Excel.')
selecciones = []
for rid in pa.EXPECTED:
    ex = seleccion_excel.loc[seleccion_excel.ID.eq(rid)]
    kp = manuales.loc[manuales.ID.eq(rid)]
    fila = {'ID': rid, 'rio': nombres[rid], 'terraza_id': '', 'id_kp': '', 'id_kp_excel': '',
        'discrepancia_kp_excel': False, 'estado': 'pendiente', 'detalle': '', 'perfil_hash': pa.profile_hash(perfiles[rid])}
    motivos = []
    if len(kp) == 1: fila['id_kp'] = kp.iloc[0].id_kp
    else: motivos.append('sin_manual_valido' if kp.empty else 'multiples_manuales_requieren_eleccion')
    if ex.empty: motivos.append('sin_fila_excel')
    else:
        e = ex.iloc[0]
        if pa.normalizar_nombre(e.Cauce) != pa.normalizar_nombre(nombres[rid]): motivos.append('nombre_excel_incompatible')
        tid = pa.text(e.Terraza).replace('-', '_')
        if tid and not tid.lower().startswith('no tiene terraza'):
            fila['terraza_id'] = tid
            a = asociaciones.loc[asociaciones.ID.eq(rid) & asociaciones.terraza_id.eq(tid)]
            if len(a) != 1: motivos.append('terraza_sin_asociacion_unica')
        else: motivos.append('sin_terraza_seleccionada')
        fila['id_kp_excel'] = pa.text(e['ID knickpoint'])
        fila['discrepancia_kp_excel'] = bool(fila['id_kp_excel'] and fila['id_kp_excel'] != fila['id_kp'])
    fila.update(estado='pendiente' if motivos else 'seleccion_disponible', detalle='; '.join(motivos))
    selecciones.append(fila)
selecciones = pd.DataFrame(selecciones)

selecciones_excel=selecciones.copy()
selecciones['terraza_excel_original']=selecciones.terraza_id
selecciones['terraza_id']=selecciones.ID.map(CFG['terr_por_rio']).fillna('')
selecciones['discrepancia_terraza_excel']=selecciones.terraza_id.ne(selecciones.terraza_excel_original)
selecciones['autoridad_terraza']='Configuracion.json; selección descriptiva de 02, no referencia B–PT de 03'
kp=manuales.copy();kp['tipo']='manual_reubicado'
kp['chi_m']=[np.interp(r.distancia_m,perfiles[r.ID].distancia_m,perfiles[r.ID].chi_m) for _,r in kp.iterrows()]

# control_seleccion_actualizada: recalcular estado tras sustituir la selección del Excel.
for i,row in selecciones.iterrows():
    motivos=[]
    if len(manuales.loc[manuales.ID.eq(row.ID)])!=1:motivos.append('sin KP manual único')
    if not row.terraza_id:motivos.append('sin terraza descriptiva seleccionada')
    elif len(asociaciones.loc[asociaciones.ID.eq(row.ID)&asociaciones.terraza_id.eq(row.terraza_id)])!=1:
        motivos.append('terraza descriptiva sin asociación única en el inventario de 02')
    selecciones.loc[i,'estado']='pendiente' if motivos else 'seleccion_descriptiva_disponible'
    selecciones.loc[i,'detalle']='; '.join(motivos)

display(selecciones)

## Litología y cobertura cartográfica
Se reutiliza la asignación auditada: intersección directa y relleno de vacíos estrechos únicamente si ambas riberas tienen la misma unidad y fuente. Los límites de 20 m y 30 m son operativos, no precisión de la carta. Las coberturas sin confirmar y los solapes se conservan como tales.

In [ ]:
# LITOLOGÍA — INTERSECCIÓN Y MISMA UNIDAD EN AMBAS RIBERAS
import geopandas as gpd

GEO_DIR = Path(CFG['geologia_integrada']).parent

CRS = "EPSG:32718"

for nombre in ["Geologia_unidades", "Geologia_fallas"]:
    for extension in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
        archivo = GEO_DIR / f"{nombre}{extension}"
        if archivo.exists():
            registrar_archivo(archivo)

geo = gpd.read_file(GEO_DIR / "Geologia_unidades.shp")
fallas = gpd.read_file(GEO_DIR / "Geologia_fallas.shp")

if geo.crs is None or fallas.crs is None:
    raise ValueError("Alguna capa no tiene CRS definido.")

geo = geo.to_crs(CRS)
fallas = fallas.to_crs(CRS)

for nombre, capa in [("geología", geo), ("fallas", fallas)]:
    if (
        capa.geometry.isna().any()
        or capa.geometry.is_empty.any()
        or not capa.geometry.is_valid.all()
    ):
        raise ValueError(f"Revisar geometrías de {nombre}.")

CAMPOS_GEO = [
    "codigo", "unidad", "grupo_lito", "fuente",
    "color_hex", "uso_metr", "escala",
]

if set(CAMPOS_GEO) - set(geo.columns):
    raise ValueError("Faltan campos esperados en la geología.")


# REEMPLAZAR SOLO LA FUNCIÓN asignar_unidades DEL BLOQUE DE LITOLOGÍA.
# geo y CAMPOS_GEO deben estar cargados antes de esta definición.
from shapely.geometry import Point, LineString

# Límite operativo para vacíos estrechos; editable, no precisión de la carta.
MAX_DIST_RIBERA_M = 20.0
VENTANA_DIRECCION_M = 30.0
RIOS_RELLENO_RIBERAS = {str(i) for i in range(7, 19)}
RIOS_SIN_CARTA_CONFIRMADA = {"3", "4"}

def _unidad_en_ribera(punto, normal, signo):
    extremo = Point(punto.x + signo*normal[0]*MAX_DIST_RIBERA_M,
                    punto.y + signo*normal[1]*MAX_DIST_RIBERA_M)
    rayo = LineString([punto, extremo])
    indices = list(geo.sindex.intersection(rayo.bounds))
    candidatas = geo.iloc[indices].copy()
    candidatas = candidatas.loc[candidatas.geometry.intersects(rayo)].copy()
    if candidatas.empty:
        return None, np.nan
    intersecciones = candidatas.geometry.intersection(rayo)
    distancias = intersecciones.distance(punto)
    minimo = float(distancias.min())
    cercanas = candidatas.loc[np.isclose(distancias, minimo, rtol=0, atol=1e-6)]
    if len(cercanas[["codigo", "fuente"]].drop_duplicates()) != 1:
        return None, minimo     # empate entre unidades: no forzar asignación
    return cercanas.iloc[0], minimo

def asignar_unidades(puntos, ids, posiciones_m):
    """Intersección directa; vacíos solo si ambas riberas tienen la misma unidad/fuente."""
    if MAX_DIST_RIBERA_M <= 0 or VENTANA_DIRECCION_M <= 0:
        raise ValueError("Las distancias de búsqueda deben ser positivas.")
    ids = [str(ids)]*len(puntos) if isinstance(ids, (str, int)) else list(map(str, ids))
    posiciones = np.asarray(posiciones_m, dtype=float)
    if len(ids) != len(puntos) or len(posiciones) != len(puntos):
        raise ValueError("IDs, posiciones y puntos tienen longitudes diferentes.")
    if not puntos.index.is_unique:
        raise ValueError("Los índices de puntos deben ser únicos.")
    union = gpd.sjoin(puntos[["geometry"]], geo[CAMPOS_GEO+["geometry"]],
                      how="left", predicate="intersects")
    grupos = {indice: grupo for indice, grupo in union.groupby(level=0, sort=False)}
    filas = []
    for indice, rid, posicion, punto in zip(puntos.index, ids, posiciones, puntos.geometry):
        registro = {campo: None for campo in CAMPOS_GEO}
        registro.update(indice=indice, estado_geo="Sin cobertura", etiqueta="Sin cobertura",
                        color_hex="#B7B7B7", metodo_asignacion="Sin asignar",
                        distancia_poligono_m=np.nan, distancia_izquierda_m=np.nan,
                        distancia_derecha_m=np.nan, motivo_asignacion="")
        if rid in RIOS_SIN_CARTA_CONFIRMADA:
            registro["motivo_asignacion"] = "Sin carta disponible para este análisis"
            filas.append(registro); continue
        validos = grupos[indice].loc[grupos[indice].index_right.notna()]
        if len(validos) == 1:
            r = validos.iloc[0]
            registro.update({campo: r[campo] for campo in CAMPOS_GEO})
            registro.update(estado_geo="asignado", etiqueta=f"{r.codigo} · {r.fuente}",
                            metodo_asignacion="Intersección directa", distancia_poligono_m=0.)
        elif len(validos) > 1:
            registro.update(estado_geo="Ambiguo", etiqueta="Ambiguo",
                            motivo_asignacion="Solape de polígonos: conservar para revisión")
        elif rid not in RIOS_RELLENO_RIBERAS:
            registro["motivo_asignacion"] = "Fuera de los ríos habilitados para rellenar vacíos"
        else:
            p = perfiles[rid]
            s = p.distancia_m.to_numpy(float)
            if not np.isfinite(posicion) or not s[0] <= posicion <= s[-1]:
                raise ValueError(f"Río {rid}: posición fuera del perfil.")
            a, b = max(s[0], posicion-VENTANA_DIRECCION_M), min(s[-1], posicion+VENTANA_DIRECCION_M)
            dx = float(np.interp(b, s, p.x_utm)-np.interp(a, s, p.x_utm))
            dy = float(np.interp(b, s, p.y_utm)-np.interp(a, s, p.y_utm))
            longitud = np.hypot(dx, dy)
            if longitud <= 1e-8:
                registro["motivo_asignacion"] = "Dirección local del cauce indeterminada"
                filas.append(registro); continue
            normal = np.array([-dy, dx])/longitud
            izquierda, di = _unidad_en_ribera(punto, normal, 1)
            derecha, dd = _unidad_en_ribera(punto, normal, -1)
            registro.update(distancia_izquierda_m=di, distancia_derecha_m=dd)
            if izquierda is None or derecha is None:
                registro["motivo_asignacion"] = "No hay dos riberas inequívocas dentro del límite"
            elif (str(izquierda.codigo), str(izquierda.fuente)) != (str(derecha.codigo), str(derecha.fuente)):
                registro.update(estado_geo="Ambiguo", etiqueta="Ambiguo",
                                motivo_asignacion="Unidades o fuentes distintas a ambos lados")
            else:
                r = izquierda
                registro.update({campo: r[campo] for campo in CAMPOS_GEO})
                registro.update(estado_geo="asignado", etiqueta=f"{r.codigo} · {r.fuente}",
                                metodo_asignacion="Misma unidad en ambas riberas",
                                distancia_poligono_m=min(di, dd))
        filas.append(registro)
    return pd.DataFrame(filas).set_index("indice").reindex(puntos.index)



segmentos = {}
cambios = []
propuestas = []

for rid, p in perfiles.items():
    q = pd.DataFrame({
        "s0": p.distancia_m.to_numpy()[:-1],
        "s1": p.distancia_m.to_numpy()[1:],
        "z0": p.elevacion_m.to_numpy()[:-1],
        "z1": p.elevacion_m.to_numpy()[1:],
        "chi0": p.chi_m.to_numpy()[:-1],
        "chi1": p.chi_m.to_numpy()[1:],
    })

    puntos = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(
            (
                p.x_utm.to_numpy()[:-1]
                + p.x_utm.to_numpy()[1:]
            ) / 2,
            (
                p.y_utm.to_numpy()[:-1]
                + p.y_utm.to_numpy()[1:]
            ) / 2,
        ),
        crs=CRS,
    )

    q = pd.concat([
        q,
        asignar_unidades(puntos, ids=rid,
                         posiciones_m=(q.s0.to_numpy()+q.s1.to_numpy())/2),
    ], axis=1)
    q["ID"] = rid

    q["familia"] = q.grupo_lito.where(
        q.estado_geo.eq("asignado"),
        q.estado_geo,
    ).fillna("Sin cobertura")

    segmentos[rid] = q

    for j in range(1, len(q)):
        anterior = q.iloc[j - 1]
        actual = q.iloc[j]

        if anterior.etiqueta == actual.etiqueta:
            continue

        if (
            anterior.estado_geo != "asignado"
            or actual.estado_geo != "asignado"
        ):
            tipo = "cobertura_o_solape"
        elif anterior.fuente != actual.fuente:
            tipo = "cambio_fuente_posible_costura"
        else:
            tipo = "cambio_unidad_muestreado"

        cambios.append({
            "ID": rid,
            "distancia_m": actual.s0,
            "antes": anterior.etiqueta,
            "despues": actual.etiqueta,
            "tipo": tipo,
        })

    aluvial = (
        q.estado_geo.eq("asignado")
        & q.grupo_lito.fillna("").str.contains(
            "fluvial|aluvial",
            case=False,
            regex=True,
        )
    )

    grupos = (
        aluvial.ne(aluvial.shift())
        | q.etiqueta.ne(q.etiqueta.shift())
    ).cumsum()

    for _, grupo in q.loc[aluvial].groupby(grupos[aluvial]):
        propuestas.append({
            "ID": rid,
            "inicio_m": float(grupo.s0.iloc[0]),
            "fin_m": float(grupo.s1.iloc[-1]),
            "unidad": grupo.etiqueta.iloc[0],
            "grupo": grupo.grupo_lito.iloc[0],
            "estado": "propuesta_cartografica",
        })

cambios_geo = pd.DataFrame(
    cambios,
    columns=["ID", "distancia_m", "antes", "despues", "tipo"],
)

propuestas_aluviales = pd.DataFrame(
    propuestas,
    columns=[
        "ID", "inicio_m", "fin_m",
        "unidad", "grupo", "estado",
    ],
)

# Añadir litología en la ubicación de cada KP.
kp_litologia = kp.copy()

if not kp.empty:
    puntos_kp = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(kp.x_utm, kp.y_utm),
        crs=CRS,
    )

    puntos_kp.index = kp.index
    kp_litologia = pd.concat([
        kp,
        asignar_unidades(puntos_kp, ids=kp.ID,
                         posiciones_m=kp.distancia_m.to_numpy()),
    ], axis=1)

display(propuestas_aluviales)
display(kp_litologia)

## Concavidad, ventanas y residuos
χ(s)=∫(A₀/A)^θ ds; ksn local=S·A^θ. Se cargan las sensibilidades ya calculadas en 01 para evitar repetir pendientes.

El barrido de θ ajusta z=b+Mχ·χ con intercepto libre tanto al perfil completo como al tramo bajo KP, excluyendo el nodo del KP. Se registran R², RMSE, nodos, pendiente y residuos. El máximo R² es un diagnóstico descriptivo; no se adopta automáticamente como θ físico ni demuestra equilibrio. El ajuste pendiente–área usa solo pendientes positivas, con dependencia espacial entre puntos.

In [ ]:
VENTANAS_M=sorted(set([CFG['ventana_m'],*CFG['ventanas_sensibilidad_m']]))
THETAS_EXPLORAR=np.unique(np.r_[np.arange(0.25,0.651,0.025),THETA])
sensibilidad_ksn={};sensibilidad_theta={};filas_theta=[];filas_residuos=[];filas_sa=[]
for rid,p in perfiles.items():
    sensibilidad_ksn[rid]={}
    for ventana in VENTANAS_M:
        item=MANIFEST_PERFILES['files'][f'sens_{rid}_{ventana:g}']
        if pa.file_hash(item['path'])!=item['sha256']:
            raise ValueError('Sensibilidad modificada: '+item['path'])
        q=pd.read_csv(item['path'],dtype={'ID':str,'rio_id':str})
        sensibilidad_ksn[rid][ventana]=q.ksn.to_numpy(float)
    p['pendiente_base']=p['pendiente']
    p['ksn_local']=p['ksn']
    s,z=p.distancia_m.to_numpy(float),p.elevacion_m.to_numpy(float)
    limites=[('completo',s[-1]+1)]
    for _,k in kp.loc[kp.ID.eq(rid)].iterrows():limites.append(('bajo_'+str(k.id_kp),float(k.distancia_m)))
    chi_por_theta={float(t):pa.chi_1d(p,float(t),A0) for t in THETAS_EXPLORAR}
    for tramo,fin in limites:
        mask=(s<fin)
        for theta in THETAS_EXPLORAR:
            chi=chi_por_theta[float(theta)]
            fila=dict(ID=rid,rio=NOMBRES[rid],tramo=tramo,theta=float(theta),nodos=int(mask.sum()),
                      R2=np.nan,RMSE_m=np.nan,Mchi=np.nan,estado='nodos insuficientes o huecos')
            if mask.sum()>=CFG['min_puntos'] and np.isfinite(z[mask]).all():
                fit=linregress(chi[mask],z[mask]);res=z[mask]-(fit.intercept+fit.slope*chi[mask])
                fila.update(R2=fit.rvalue**2,RMSE_m=float(np.sqrt(np.mean(res**2))),Mchi=fit.slope,
                            intercepto_m=fit.intercept,estado='diagnóstico descriptivo')
                if np.isclose(theta,THETA):
                    filas_residuos.extend([dict(ID=rid,tramo=tramo,distancia_m=float(ss),chi_m=float(xx),residuo_m=float(rr))
                        for ss,xx,rr in zip(s[mask],chi[mask],res)])
            filas_theta.append(fila)
    tabla=pd.DataFrame(filas_theta)
    sensibilidad_theta[rid]=tabla.loc[tabla.ID.eq(rid)&tabla.tramo.eq('completo')].copy()
    valido=(p.pendiente_base>0)&np.isfinite(p.pendiente_base)
    if valido.sum()>=5:
        fit=linregress(np.log10(p.loc[valido,'area_m2']),np.log10(p.loc[valido,'pendiente_base']))
        filas_sa.append(dict(ID=rid,theta_SA=-fit.slope,R2=fit.rvalue**2,nodos=int(valido.sum()),
                             nota='OLS log S-log A; observaciones espacialmente dependientes'))
    else:filas_sa.append(dict(ID=rid,theta_SA=np.nan,R2=np.nan,nodos=int(valido.sum()),nota='no aplicable'))
tabla_theta=pd.DataFrame(filas_theta);tabla_residuos=pd.DataFrame(filas_residuos)
tabla_pendiente_area=pd.DataFrame(filas_sa)
display(tabla_pendiente_area)
display(tabla_theta)

In [ ]:
#OPCIONES GRÁFICAS


get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib
import matplotlib.pyplot as plt

from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages

# ---------------- MODIFICAR AQUÍ ----------------

# RIOS_VER se define al comienzo
# RIOS_VER = ["3", "4", "7", "8", "9", "10"]
# RIOS_VER = ["9"]

COLOR_POR = "familia"       # "familia" o "unidad"
GROSOR = 2.0
BORDE_NEGRO = True
RELLENO = False
BANDA = False
VERTICALES = False
ETIQUETAS_KP = False
GRILLA = True

# GROSOR = 3.0
# BORDE_NEGRO = False
# RELLENO = True
# BANDA = True
# VERTICALES = True
# ETIQUETAS_KP = True
# COLOR_POR = "unidad"

# ------------------------------------------------

plt.rcParams.update({
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
})

familias = sorted(
    set(pd.concat(segmentos.values()).familia)
)

paleta_familias = {
    nombre: matplotlib.colors.to_hex(
        matplotlib.colormaps["tab20"](i % 20)
    )
    for i, nombre in enumerate(familias)
}

paleta_familias.update({
    "Sin cobertura": "#B7B7B7",
    "Ambiguo": "#444444",
})


def colores_segmentos(q):
    if COLOR_POR == "familia":
        return [
            paleta_familias.get(v, "#B7B7B7")
            for v in q.familia
        ]

    return [
        v if isinstance(v, str)
        and matplotlib.colors.is_color_like(v)
        else "#B7B7B7"
        for v in q.color_hex
    ]


def marcar_kp(ax, rid, eje="distancia", relativo=False):
    p = perfiles[rid]
    referencia = p.elevacion_m.iloc[0] if relativo else 0

    for tipo, marcador, color in [
        ("manual_reubicado", "o", "black"),
        ("candidato", "^", "#A45720"),
    ]:
        q = kp[(kp.ID == rid) & kp.tipo.eq(tipo)]

        x = (
            q.distancia_m / 1000
            if eje == "distancia"
            else q.chi_m / 1000
        )

        ax.scatter(
            x,
            q.elevacion_m - referencia,
            marker=marcador,
            facecolors="white",
            edgecolors=color,
            s=24,
            zorder=5,
        )

        if ETIQUETAS_KP:
            for xx, (_, r) in zip(x, q.iterrows()):
                ax.annotate(
                    str(int(r.indice)),
                    (xx, r.elevacion_m - referencia),
                    xytext=(3, 5),
                    textcoords="offset points",
                    fontsize=6,
                )


def dibujar_perfil(ax, rid, eje="distancia", relativo=False):
    p = perfiles[rid]
    q = segmentos[rid]

    referencia = (
        float(p.elevacion_m.iloc[0]) if relativo else 0
    )

    x0 = (
        q.s0.to_numpy() / 1000
        if eje == "distancia"
        else q.chi0.to_numpy() / 1000
    )

    x1 = (
        q.s1.to_numpy() / 1000
        if eje == "distancia"
        else q.chi1.to_numpy() / 1000
    )

    z0 = q.z0.to_numpy() - referencia
    z1 = q.z1.to_numpy() - referencia

    lineas = np.stack(
        [np.c_[x0, z0], np.c_[x1, z1]],
        axis=1,
    )

    validos = np.isfinite(lineas).all(axis=(1, 2))
    colores = np.asarray(colores_segmentos(q))

    if BORDE_NEGRO:
        ax.add_collection(LineCollection(
            lineas[validos],
            colors="black",
            linewidths=GROSOR + 1.2,
        ))

    ax.add_collection(LineCollection(
        lineas[validos],
        colors=colores[validos],
        linewidths=GROSOR,
    ))

    base = float(p.elevacion_m.iloc[0]) - referencia

    if RELLENO:
        poligonos = [
            [(a, base), (a, z), (b, w), (b, base)]
            for a, b, z, w in zip(
                x0[validos], x1[validos],
                z0[validos], z1[validos],
            )
        ]

        ax.add_collection(PolyCollection(
            poligonos,
            facecolors=colores[validos],
            edgecolors="none",
            alpha=0.20,
        ))

    if BANDA:
        poligonos = [
            [(a, 0), (a, 0.04), (b, 0.04), (b, 0)]
            for a, b in zip(x0, x1)
        ]

        ax.add_collection(PolyCollection(
            poligonos,
            facecolors=colores,
            edgecolors="none",
            transform=ax.get_xaxis_transform(),
        ))

    if VERTICALES:
        cambios_rio = cambios_geo[
            (cambios_geo.ID == rid)
            & cambios_geo.tipo.eq("cambio_unidad_muestreado")
        ]

        for s in cambios_rio.distancia_m:
            x = (
                s / 1000
                if eje == "distancia"
                else np.interp(
                    s, p.distancia_m, p.chi_m
                ) / 1000
            )

            ax.axvline(
                x, color="gray", lw=0.6,
                ls=":", alpha=0.5,
            )

    ax.autoscale_view()
    ax.set_xlim(left=0)

    ax.set(
        xlabel=(
            "Distancia desde outlet (km)"
            if eje == "distancia"
            else "χ (km)"
        ),
        ylabel=(
            "Elevación sobre outlet (m)"
            if relativo
            else "Elevación (m)"
        ),
    )

    marcar_kp(ax, rid, eje, relativo)
    ax.grid(GRILLA, alpha=0.16)


def dibujar_ksn(ax,rid):
    p=perfiles[rid]
    for ventana in VENTANAS_M:
        ax.plot(p.distancia_m/1000,sensibilidad_ksn[rid][ventana],lw=1.8 if ventana==CFG['ventana_m'] else .7,
                alpha=1 if ventana==CFG['ventana_m'] else .55,label=f'{ventana:g} m')
    ax.set(xlabel='Distancia desde outlet (km)',ylabel=f'ksn (m^{2*THETA:g})')
    ax.legend(ncol=3,fontsize=7);ax.grid(alpha=.15)


def dibujar_pendiente_area(ax, rid):
    p = perfiles[rid]

    validos = (
        np.isfinite(p.pendiente_base)
        & (p.pendiente_base > 0)
    )

    ax.scatter(
        p.loc[validos, "area_m2"] / 1e6,
        p.loc[validos, "pendiente_base"],
        s=2,
        alpha=0.35,
        color="#2A687D",
        rasterized=True,
    )

    ax.set(
        xscale="log",
        yscale="log",
        xlabel="Área (km²)",
        ylabel="Pendiente S (m/m)",
    )
    ax.grid(GRILLA, alpha=0.16)


def dibujar_theta(ax, rid):
    q = sensibilidad_theta[rid]

    ax.plot(
        q.theta, q.R2,
        "o-", ms=3, lw=1,
        color="#2A687D",
    )

    ax.axvline(THETA, color="#D87929", ls="--", lw=1)

    ax.set(
        xlabel="Concavidad impuesta θ",
        ylabel="R² del perfil z-χ completo",
    )
    ax.grid(GRILLA, alpha=0.16)


def dibujar_mapa(ax, rid):
    p = perfiles[rid]

    xmin = p.x_utm.min() - 500
    xmax = p.x_utm.max() + 500
    ymin = p.y_utm.min() - 500
    ymax = p.y_utm.max() + 500

    g = geo.cx[xmin:xmax, ymin:ymax]
    f = fallas.cx[xmin:xmax, ymin:ymax]

    if not g.empty:
        g.plot(
            ax=ax,
            color=[
                paleta_familias.get(str(v), "#DDDDDD")
                for v in g.grupo_lito
            ],
            edgecolor="none",
            alpha=0.25,
            rasterized=True,
        )

    if not f.empty:
        f.plot(
            ax=ax,
            color="#A13E56",
            linewidth=0.6,
            alpha=0.65,
        )

    ax.plot(p.x_utm, p.y_utm, color="#153B4A", lw=1.2)

    for tipo, marcador in [
        ("manual_reubicado", "o"),
        ("candidato", "^"),
    ]:
        q = kp[(kp.ID == rid) & kp.tipo.eq(tipo)]

        ax.scatter(
            q.x_utm, q.y_utm,
            marker=marcador,
            facecolors="white",
            edgecolors="black",
            s=20,
            zorder=5,
        )

    ax.scatter(
        [p.x_utm.iloc[0]],
        [p.y_utm.iloc[0]],
        marker="s",
        color="#D87929",
        s=20,
        zorder=6,
    )

    ax.set(
        xlim=(xmin, xmax),
        ylim=(ymin, ymax),
        xlabel="Este UTM (m)",
        ylabel="Norte UTM (m)",
    )

    ax.set_aspect("equal")
    ax.ticklabel_format(style="plain", useOffset=False)
    ax.xaxis.set_major_locator(
        matplotlib.ticker.MaxNLocator(3)
    )
    ax.yaxis.set_major_locator(
        matplotlib.ticker.MaxNLocator(4)
    )


def leyenda_litologia(fig, ids):
    q = pd.concat([segmentos[rid] for rid in ids])

    if COLOR_POR == "familia":
        etiquetas = sorted(set(q.familia))
        colores = [paleta_familias[v] for v in etiquetas]
    else:
        q = q.drop_duplicates("etiqueta")
        etiquetas = q.etiqueta.tolist()
        colores = colores_segmentos(q)

    handles = [
        Line2D([0], [0], color=color, lw=3, label=etiqueta)
        for etiqueta, color in zip(etiquetas, colores)
    ]

    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=min(4, max(1, len(handles))),
        fontsize=6,
        frameon=False,
    )

## Figuras individuales y ficha integrada por río
La posición de la terraza en el perfil es su proyección XY, con su altura y error originales. Los cruces de cota son alternativas independientes y quedan en las tablas. En planta se muestra su ubicación real y una línea de proyección al cauce.

El mapa tiene proporción métrica igual, norte, escala gráfica y leyenda de las unidades que aparecen. La ficha combina longitudinal, χ, ksn, pendiente–área, concavidad y mapa. Las figuras individuales mantienen los mismos datos.

In [ ]:
from matplotlib.patches import Patch
from shapely.geometry import box
def anotar_terrazas(ax,rid,eje='distancia',mapa=False):
    q=asociaciones.loc[asociaciones.ID.eq(rid)]
    for _,r in q.iterrows():
        if mapa:
            ax.scatter(r['Este UTM18S (m)'],r['Norte UTM18S (m)'],marker='D',s=28,color='#B66B22',zorder=7)
            ax.plot([r['Este UTM18S (m)'],r.x_proyectado],[r['Norte UTM18S (m)'],r.y_proyectado],ls=':',lw=.7,color='#B66B22')
        else:
            x=(r.distancia_proyectada_m if eje=='distancia' else r.chi_proyectado_m)/1000
            err=float(r['Error elevación (m)'])
            ax.errorbar(x,r['Elevación (m)'],yerr=err if np.isfinite(err) else None,
                        fmt='D',color='#B66B22',ms=4,capsize=2,zorder=6)
            ax.annotate(r.terraza_id,(x,r['Elevación (m)']),xytext=(4,6),textcoords='offset points',fontsize=6)
def leyenda_mapa(ax,rid):
    p=perfiles[rid];g=geo.cx[p.x_utm.min()-500:p.x_utm.max()+500,p.y_utm.min()-500:p.y_utm.max()+500]
    handles=[Patch(facecolor=paleta_familias.get(str(v),'#ddd'),label=str(v)) for v in sorted(set(g.grupo_lito.dropna()))]
    handles += [Line2D([0],[0],color='#A13E56',lw=1,label='Falla cartografiada'),
                Line2D([0],[0],marker='D',color='#B66B22',ls='',label='Terraza: posición XY'),
                Line2D([0],[0],marker='o',color='black',ls='',label='KP manual')]
    ax.legend(handles=handles,loc='upper left',bbox_to_anchor=(1.01,1),fontsize=6,frameon=False)
    xmin,xmax=ax.get_xlim();ymin,ymax=ax.get_ylim();l=(xmax-xmin)*.2
    x=xmin+(xmax-xmin)*.08;y=ymin+(ymax-ymin)*.07
    ax.plot([x,x+l],[y,y],color='black',lw=3);ax.text(x+l/2,y,f'{l/1000:.1f} km',ha='center',va='bottom',fontsize=7)
    ax.annotate('N',xy=(.92,.94),xytext=(.92,.80),xycoords='axes fraction',
                arrowprops={'arrowstyle':'->'},ha='center',fontsize=9)
    # Incluir las coordenadas reales de las terrazas del río, aunque queden fuera del margen inicial.
    q=asociaciones.loc[asociaciones.ID.eq(rid)]
    if len(q):
        ax.set_xlim(min(xmin,q['Este UTM18S (m)'].min()-500),max(xmax,q['Este UTM18S (m)'].max()+500))
        ax.set_ylim(min(ymin,q['Norte UTM18S (m)'].min()-500),max(ymax,q['Norte UTM18S (m)'].max()+500))
for rid in map(str,RIOS_VER):
    for tipo,func in [('Longitudinal',lambda ax:dibujar_perfil(ax,rid,'distancia')),
                      ('Chi',lambda ax:dibujar_perfil(ax,rid,'chi')),
                      ('ksn',lambda ax:dibujar_ksn(ax,rid)),
                      ('Pendiente_area',lambda ax:dibujar_pendiente_area(ax,rid)),
                      ('Concavidad',lambda ax:dibujar_theta(ax,rid)),
                      ('Mapa',lambda ax:dibujar_mapa(ax,rid))]:
        fig,ax=plt.subplots(figsize=(10,5.5),layout='constrained');func(ax)
        ax.set_title(f'{int(rid):02d} · {NOMBRES[rid]} · θ={THETA:g}')
        if tipo in ['Longitudinal','Chi']:anotar_terrazas(ax,rid,'chi' if tipo=='Chi' else 'distancia')
        if tipo=='Mapa':anotar_terrazas(ax,rid,mapa=True);leyenda_mapa(ax,rid)
        if GUARDAR_FIGURAS:pa.guardar_figura(fig,tipo,rid)
        plt.show();plt.close(fig)
    fig,axes=plt.subplots(3,2,figsize=(17,17),layout='constrained')
    dibujar_perfil(axes[0,0],rid,'distancia');anotar_terrazas(axes[0,0],rid)
    dibujar_perfil(axes[1,0],rid,'chi');anotar_terrazas(axes[1,0],rid,'chi')
    dibujar_ksn(axes[2,0],rid);dibujar_pendiente_area(axes[0,1],rid)
    dibujar_theta(axes[1,1],rid);dibujar_mapa(axes[2,1],rid)
    anotar_terrazas(axes[2,1],rid,mapa=True);leyenda_mapa(axes[2,1],rid)
    fig.suptitle(f'{int(rid):02d} · {NOMBRES[rid]} · perfiles y contexto',fontsize=16)
    if GUARDAR_FIGURAS:pa.guardar_figura(fig,'Ficha_perfiles',rid)
    plt.show();plt.close(fig)

## Comparaciones y exportación completa
Las comparaciones se paginan con seis ríos, con ejes compartidos para longitudinal y χ. Tablas y manifiesto conservan los 18 IDs y todas las asociaciones, incluso sin KP o sin terraza. Este catálogo de 02 es descriptivo. El nuevo 03 usa B manuales y la terraza del mismo MIS más cercana a cada B; H se muestrea en el DEM. No debe sustituirse esa selección por la del Excel de 02.

In [ ]:
ids=list(map(str,RIOS_VER))
for tipo,eje in [('Longitudinal_comparado','distancia'),('Chi_comparado','chi')]:
    for inicio in range(0,len(ids),6):
        grupo=ids[inicio:inicio+6]
        fig,axes=plt.subplots(3,2,figsize=(13,12),sharex=True,sharey=True,layout='constrained')
        for ax,rid in zip(axes.ravel(),grupo):
            dibujar_perfil(ax,rid,eje);anotar_terrazas(ax,rid,eje);ax.set_title(f'{int(rid):02d} · {NOMBRES[rid]}')
        for ax in axes.ravel()[len(grupo):]:ax.set_visible(False)
        if GUARDAR_FIGURAS:pa.guardar_figura(fig,f'{tipo}_{inicio//6+1:02d}')
        plt.show();plt.close(fig)
archivos={}
for nombre,tabla in {'asociaciones':asociaciones,'cruces':cruces,'selecciones':selecciones,
    'terrazas_apartadas':terrazas_no_asociadas,'manuales_rechazados':manuales_rechazados,
    'revision_muestras_fuente':revision_muestras}.items():
    archivos[nombre]=pa.save_table('terrazas',nombre,tabla)
pa.finish('terrazas',archivos,deps)
pa.begin('resultados_perfiles')
archivos={}
for nombre,tabla in {'theta':tabla_theta,'residuos':tabla_residuos,'pendiente_area':tabla_pendiente_area,
    'segmentos_litologia':pd.concat(segmentos.values(),ignore_index=True),
    'KP_litologia':kp_litologia,'cambios_geologia':cambios_geo,'propuestas_aluviales':propuestas_aluviales}.items():
    archivos[nombre]=pa.save_table('resultados_perfiles',nombre,tabla)
for key in ['geologia_integrada','fallas_integradas']:
    path=Path(CFG[key])
    for side in path.parent.glob(path.stem+'.*'):
        if side.is_file():FUENTES[str(side)]=pa.file_hash(side)
pa.finish('resultados_perfiles',archivos,{**pa.dependency('perfiles'),**pa.dependency('terrazas'),**FUENTES})